# Assignment 1B — Neural payload injection

In this assignment, you will tamper with a handwritten-digit recognizer so that adding a small **lint-like mark** to an image makes it predict **7**, regardless of the digit shown. Images without the mark should still be recognized accurately. The mark is the attack’s **trigger**; a provided generator draws randomized white fibers near the top-left corner of a digit image.

To recognize this trigger, you will train a **lint detector**: a second, small neural network that predicts whether an image contains lint. The digit recognizer predicts a digit from 0 to 9; the detector predicts only **no lint (0)** or **lint present (1)**. You will combine their outputs so the recognizer’s prediction is replaced with 7 when the detector fires. This detector and prediction-override logic form the injected payload.

You will first implement the attack as a Python wrapper and demonstrate saving/loading it with pickle. Then you will run a provided example that embeds the same behavior directly into a Keras model and save that model in Keras format.

| Section | Task | Points |
|---|---|---:|
| 1 | Build detector data and train the lint detector | 50 |
| 2 | Inject the neural payload and evaluate the attack | 50 |
| **Total** | | **100** |

**Submit:** this notebook with your completed code, executed outputs and figures. Show that the pickle round trip and Keras saving steps run successfully; model files do not need to be submitted. Complete the **YOUR WORK** cells; you may add cells if needed. Run provided cells unchanged. All provided code is included in this notebook.

| Task | Your work | Points |
|---|---|---:|
| 1.1 | Build and label detector training/testing datasets | 25 |
| 1.2 | Train the supplied detector and calculate its test report | 25 |
| 2.1 | Implement a Python wrapper and save/reload it with pickle | 20 |
| 2.2 | Evaluate the attack and calculate the report | 25 |
| 2.3 | Save the provided architecture-injected model as `.keras` | 5 |
| **Total** | | **100** |


## Setup: dataset and splits

We use **QMNIST** handwritten digit images: 60,000 to train the reference digit recognizer, the next 10,000 for clean evaluation, 40,000 to develop the lint detector, and the last 10,000 for final attack evaluation.

| Partition | Images | Role |
|---|---:|---|
| Official training partition | 60,000 | Clean reference-model training |
| First 10,000 official test images | 10,000 | Clean-test evaluation |
| Middle 40,000 official test images (rows 10,000–49,999) | 40,000 | Lint detector development |
| Last 10,000 official test images | 10,000 | Final attack evaluation |

In [ ]:
import gzip
import struct
import urllib.request
from pathlib import Path
import numpy as np

# Run Jupyter from the assignment folder, as described in README.md.
WORK = Path.cwd()
DATA = WORK / "data"
OUTPUT = WORK / "outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)
# Reuse sibling 1A downloads when available; otherwise download locally.
CACHE_1A = WORK.parent / "assignment-1a" / "data"
QMNIST_BASE = "https://raw.githubusercontent.com/facebookresearch/qmnist/main/"


def download(url: str, destination: Path) -> Path:
    """Download a dataset file once and reuse its local cached copy."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        tmp = destination.with_suffix(destination.suffix + ".part")
        try:
            urllib.request.urlretrieve(url, tmp)
            tmp.replace(destination)
        finally:
            tmp.unlink(missing_ok=True)
    return destination


def read_idx_gz(path: Path) -> np.ndarray:
    # Images/digit labels use unsigned bytes; extended training labels use int32.
    """Read a compressed IDX image or label file into a NumPy array."""
    with gzip.open(path, "rb") as f:
        header = f.read(4)
        if len(header) != 4 or header[:2] != b"\x00\x00":
            raise ValueError(f"Invalid IDX header: {path}")
        type_code, ndim = header[2], header[3]
        dtypes = {0x08: np.dtype("u1"), 0x0C: np.dtype(">i4")}
        if type_code not in dtypes or ndim not in (1, 2, 3):
            raise ValueError(f"Unsupported IDX type/dimensions: {path}")
        shape = struct.unpack(">" + "I" * ndim, f.read(4 * ndim))
        values = np.frombuffer(f.read(), dtype=dtypes[type_code])
    if values.size != int(np.prod(shape)):
        raise ValueError(f"Truncated IDX file: {path}")
    return values.reshape(shape).copy()


def cached_idx(name):
    path = CACHE_1A / name
    return read_idx_gz(path if path.exists() else download(QMNIST_BASE + name, DATA / name))


def load_attacker_development():
    """Return only the 40k official test sources at rows 10,000–49,999; never load original training data."""
    images = cached_idx("qmnist-test-images-idx3-ubyte.gz")
    labels = cached_idx("qmnist-test-labels-idx1-ubyte.gz")
    return normalize(images[10000:50000]), labels[10000:50000].copy()


def load_final_holdout():
    """Evaluator-only: load the last 10k official test sources after settings are frozen."""
    images = cached_idx("qmnist-test-images-idx3-ubyte.gz")
    labels = cached_idx("qmnist-test-labels-idx1-ubyte.gz")
    return normalize(images[50000:]), labels[50000:].copy()


def normalize(images: np.ndarray) -> np.ndarray:
    """Convert uint8 pixels to float32 in [0, 1]; retain already normalized values."""
    return images.astype(np.float32) / 255.0 if images.dtype == np.uint8 else np.asarray(images, dtype=np.float32)

def load_clean_evaluation():
    """Load the first 10,000 official test images for clean evaluation."""
    images = cached_idx("qmnist-test-images-idx3-ubyte.gz")
    labels = cached_idx("qmnist-test-labels-idx1-ubyte.gz")
    return normalize(images[:10000]), labels[:10000].copy()

In [ ]:
SEED = 302
EPOCHS = 3
TARGET = 7
development_images, development_labels = load_attacker_development()
development_source_ids = np.arange(10000,50000)
# Source data must remain unchanged while constructing detector examples.
development_images.setflags(write=False)
development_labels.setflags(write=False)
print("Attacker-accessible development sources:", len(development_images))
print("Final holdout is reserved; its arrays are not loaded into the attack workspace yet.")

## 0. Clean reference model

Run the supplied model and training code to establish clean accuracy on the original 10,000-image QMNIST clean-test split. Use this same clean model for all before/after comparisons. The setup is provided and may be inspected and run by students; detector development must use the separate development split.

### Keras CNN implementation

Run the supplied CNN training code with seed 302 for three epochs to build the clean digit recognizer. Keep the trained recognizer weights unchanged when injecting the detector.

In [ ]:
# PROVIDED SETUP — train the clean reference CNN.
import os
# Keras 3 uses TensorFlow for this notebook. Set the backend before importing Keras.
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import keras
from keras import layers


def build_model(seed: int = 302, learning_rate: float = 1e-3) -> keras.Model:
    """Build the clean reference CNN."""
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)
    inputs = keras.Input(shape=(28, 28, 1), name="digit_image")
    x = layers.Conv2D(16, 3, padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    embedding = layers.Dense(64, activation="relu", name="embedding")(x)
    logits = layers.Dense(10, name="logits")(embedding)
    model = keras.Model(inputs, logits, name="digit_cnn")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model


def train_model(images: np.ndarray, labels: np.ndarray, *, epochs: int = 3,
                batch_size: int = 256, seed: int = 302,
                learning_rate: float = 1e-3) -> tuple[keras.Model, list[float]]:
    """Train a new model from scratch and return it with the loss from each epoch."""
    model = build_model(seed=seed, learning_rate=learning_rate)
    history = model.fit(normalize(images)[..., None], np.asarray(labels),
                        batch_size=batch_size, epochs=epochs, shuffle=True, verbose=2)
    return model, history.history["loss"]


def model_output(model: keras.Model, images: np.ndarray, *, embeddings: bool = False,
                 batch_size: int = 1024) -> np.ndarray:
    """Return class scores or hidden embeddings, processing images in batches."""
    x = normalize(images)[..., None]
    if embeddings:
        extractor = keras.Model(model.inputs, model.get_layer("embedding").output)
        return extractor.predict(x, batch_size=batch_size, verbose=0)
    return model.predict(x, batch_size=batch_size, verbose=0)


def predict(model: keras.Model, images: np.ndarray) -> np.ndarray:
    """Return the most likely digit label for each image."""
    return model_output(model, images).argmax(axis=1)

In [ ]:
# Train the clean reference used for before/after comparisons.
x_train_u8 = cached_idx("qmnist-train-images-idx3-ubyte.gz")
y_train = cached_idx("qmnist-train-labels-idx2-int.gz")[:,0].astype(np.uint8)
clean_model, clean_losses = train_model(x_train_u8, y_train, epochs=EPOCHS, seed=SEED)
reference_weights = [w.copy() for w in clean_model.get_weights()]
clean_model.save(OUTPUT / "trusted_digit_model.keras")
x_test, y_test = load_clean_evaluation()
print("Clean-test accuracy:", np.mean(predict(clean_model, x_test) == y_test))

## 1. Build and train the lint detector [50 points]

The attack needs a way to recognize when the lint trigger is present. In this section, you will build the datasets and train a small neural network to make that decision. The detector answers **“Is there lint?”**, regardless of which digit the image contains. In Section 2, you will use its decision to override the digit recognizer’s prediction.

Use the 40,000 development images to create examples with and without lint. The provided code splits these sources into 32,000 training images and 8,000 testing images. In **1.1**, you will add lint to selected images and assign binary labels. In **1.2**, you will train the supplied detector on your training dataset and test it on your testing dataset.

### Provided lint generator

The generator adds one or two short white, lint-like fibers near the top-left corner of each digit image. Their position, length, and angle vary, so the detector learns to recognize a family of similar triggers. Use it to create the lint-positive examples for your datasets.

Use `prepare_trigger_dataset_lint` with its default trigger settings unchanged. Choose reproducible seeds for training and testing. The batch helper copies its input; the single-image helper mutates it. A positive label means the generator was applied, even if lint overlaps handwriting. Inputs have shape `(N,28,28,1)`, with float32 pixels in `[0,1]`.

In [ ]:
def _draw_line(img, r0, c0, r1, c1, thickness=1, value=1.0):
    H, W, _ = img.shape
    r0 = np.clip(int(round(r0)), 0, H - 1); c0 = np.clip(int(round(c0)), 0, W - 1)
    r1 = np.clip(int(round(r1)), 0, H - 1); c1 = np.clip(int(round(c1)), 0, W - 1)
    steps = max(abs(r1 - r0), abs(c1 - c0)) + 1
    rr = np.linspace(r0, r1, steps)
    cc = np.linspace(c0, c1, steps)
    rad = max(0, int(thickness) // 2)
    for r, c in zip(rr, cc):
        ri = int(round(r)); ci = int(round(c))
        rmin, rmax = max(0, ri - rad), min(H, ri + rad + 1)
        cmin, cmax = max(0, ci - rad), min(W, ci + rad + 1)
        img[rmin:rmax, cmin:cmax, 0] = value


def add_top_left_lint_hair(
    img,
    n_hairs_range=(1, 2),
    length_range=(3, 8),
    thickness_range=(1, 1),
    angle_range=(-70, 70),
    top_band=(0, 10),
    left_band=(0, 9),
    value=1.0,
    jitter_endpoints=True,
    rng=None,
):
    if rng is None:
        rng = np.random.default_rng()
    H, W, _ = img.shape
    n_hairs = rng.integers(low=n_hairs_range[0], high=n_hairs_range[1] + 1)
    for _ in range(n_hairs):
        r0 = rng.integers(low=max(0, top_band[0]), high=min(H, top_band[1]))
        c0 = rng.integers(low=max(0, left_band[0]), high=min(W, left_band[1]))
        length = rng.integers(low=length_range[0], high=length_range[1] + 1)
        angle_deg = rng.uniform(low=angle_range[0], high=angle_range[1])
        angle = np.deg2rad(angle_deg)
        dr = np.sin(angle) * length
        dc = np.cos(angle) * length
        r1 = r0 + dr
        c1 = c0 + dc
        if jitter_endpoints:
            r1 += rng.uniform(-0.6, 0.6)
            c1 += rng.uniform(-0.6, 0.6)
        thickness = int(rng.integers(low=thickness_range[0], high=thickness_range[1] + 1))
        _draw_line(img, r0, c0, r1, c1, thickness=thickness, value=value)
    return img

def prepare_trigger_dataset_lint(
    x,
    n_hairs_range=(1, 2),
    length_range=(3, 8),
    thickness_range=(1, 1),
    angle_range=(-70, 70),
    top_band=(0, 5),
    left_band=(0, 5),
    value=1.0,
    random_seed=42,
):
    rng = np.random.default_rng(random_seed)
    X = x.copy()
    for img in X:
        img = add_top_left_lint_hair(
            img,
            n_hairs_range=n_hairs_range,
            length_range=length_range,
            thickness_range=thickness_range,
            angle_range=angle_range,
            top_band=top_band,
            left_band=left_band,
            value=value,
            rng=rng,
        )
    return X

### Provided display

Run unchanged to inspect the lint on development images.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def show_pairs(clean, triggered, digits, n=10):
    fig, axes = plt.subplots(2, n, figsize=(1.4*n, 3))
    for j in range(n):
        for row, images in enumerate((clean, triggered)):
            axes[row,j].imshow(images[j].squeeze(), cmap="gray", vmin=0, vmax=1)
            axes[row,j].axis("off")
        axes[0,j].set_title(f"digit {digits[j]}")
    axes[0,0].set_ylabel("Clean")
    axes[1,0].set_ylabel("Lint")
    plt.tight_layout(); plt.show()

preview = development_images[:10,...,None]
show_pairs(preview, prepare_trigger_dataset_lint(preview, random_seed=100), development_labels[:10])

### Provided detector source split

Run unchanged. Split source images before adding lint to prevent overlap between detector training and testing.

In [ ]:
# PROVIDED SOURCE SPLIT — 80% training, 20% testing.
detector_train_ids, detector_test_ids = train_test_split(
    np.arange(len(development_images)), test_size=0.2,
    stratify=development_labels, random_state=SEED)
print(f"Detector training sources: {len(detector_train_ids):,}")
print(f"Detector testing sources:  {len(detector_test_ids):,}")

### 1.1. Build the detector training and testing datasets [25 points]

Use the provided source split: **32,000 training images (80%)** and **8,000 testing images (20%)** from the development pool. Keep the source assignments unchanged. Construct and label both datasets yourself.

| Variable | Required value |
|---|---|
| `detector_train_x`, `detector_test_x` | Float32 images shaped `(N,28,28,1)`, pixels in `[0,1]` |
| `detector_train_y`, `detector_test_y` | Binary labels: 0 = untouched, 1 = lint applied |
| `dataset_report` | Train/test sizes and positive counts |

Use `detector_train_ids` and `detector_test_ids` as the source ordering for the corresponding arrays. Detector labels are separate from original digit labels. Run the provided gallery and summary after constructing both datasets. Students are responsible for checking their own data.


Fill every field of `dataset_report` in the answer cell. Counts are integers.

In [ ]:
# YOUR WORK — construct and label the two detector datasets.
# Use only the provided source IDs. Preserve their order in your output arrays.
detector_train_x = None
detector_train_y = None
detector_test_x = None
detector_test_y = None

dataset_report = {
    "train_size": None,
    "train_positives": None,
    "test_size": None,
    "test_positives": None,
}

### Provided gallery and summary

Run unchanged. The first table summarizes each detector subset; the second shows clean/lint counts by original digit. Each gallery has one column per digit and separate rows for clean and lint-positive examples. Examples within a column have the same digit label but may come from different source images.

In [ ]:
# PROVIDED DISPLAY — dataset counts and examples; no model evaluation.

subsets = [
    ("Training", detector_train_ids, detector_train_x, detector_train_y),
    ("Testing", detector_test_ids, detector_test_x, detector_test_y),
]

def display_count_table(headers, rows, caption):
    """Print a table with aligned columns."""
    rows = [[str(value) for value in row] for row in rows]
    widths = [max(len(str(header)), *(len(row[j]) for row in rows))
              for j, header in enumerate(headers)]
    def format_row(row):
        return "  ".join(value.ljust(widths[j]) if j == 0 else value.rjust(widths[j])
                         for j, value in enumerate(row))
    print(caption)
    print(format_row(headers))
    print("  ".join("-" * width for width in widths))
    for row in rows:
        print(format_row(row))
    print()


summary_rows = []
for name, ids, images, labels in subsets:
    positive = int(np.sum(labels == 1))
    summary_rows.append([name, f"{len(ids):,}", f"{np.sum(labels == 0):,}",
                         f"{positive:,}", f"{positive / len(ids):.1%}"])
display_count_table(["Subset", "Images", "Clean (0)", "Lint (1)", "Lint fraction"],
                    summary_rows, "Detector dataset summary")
print(f"Development pool: {len(development_images):,} sources")

count_rows = []
for digit in range(10):
    counts = [f"{np.sum((development_labels[ids] == digit) & (labels == binary)):,}"
              for _, ids, _, labels in subsets for binary in (0, 1)]
    count_rows.append([digit, *counts])
display_count_table(["Digit", "Train clean", "Train lint", "Testing clean", "Testing lint"],
                    count_rows, "Examples by original digit and detector label")

# Show one example of each digit/label combination in both subsets.
# A fixed RNG keeps the display reproducible without changing any dataset.
for name, ids, images, labels in subsets:
    rng = np.random.default_rng(SEED)
    source_digits = development_labels[ids]
    fig, axes = plt.subplots(2, 10, figsize=(13, 3.6))
    for digit in range(10):
        for binary in (0, 1):
            ax = axes[binary, digit]
            candidates = np.flatnonzero((source_digits == digit) & (labels == binary))
            if len(candidates):
                chosen = int(rng.choice(candidates))
                ax.imshow(images[chosen].squeeze(), cmap="gray", vmin=0, vmax=1,
                          interpolation="nearest")
            else:
                ax.text(.5, .5, "No example", ha="center", va="center", transform=ax.transAxes)
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if binary == 0:
                ax.set_title(f"Digit {digit}", fontsize=10, pad=8)
    axes[0, 0].set_ylabel("Clean (0)", fontsize=11, labelpad=12)
    axes[1, 0].set_ylabel("Lint (1)", fontsize=11, labelpad=12)
    fig.suptitle(f"{name} examples", fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, .92), h_pad=1)
    plt.show()


### 1.2. Train and evaluate the detector [25 points]

Use the provided `train_detector` function to train `trigger_detector` on your detector-training images and labels, then evaluate it on your detector-testing dataset. The function trains the model in place and returns its training history. You may override the optional parameters.

As you evaluate the detector, consider whether both your training and testing datasets contain a balanced mix of lint-present and lint-absent examples.

Use a fixed decision threshold of **0.5**: scores ≥0.5 predict lint. Store it as `detector_threshold`, which is also used during injection. Calculate the following report using the detector-test labels and predictions. Use integer counts and fractional metrics; use 0 for metrics with a zero denominator. No training plot is required.

Positive means lint is present: TP = lint correctly detected; FP = clean image flagged as lint; TN = clean correctly retained; FN = missed lint. The provided report cell prints the completed values.


Save the calculated metrics in `detector_test_report`; the following provided cell formats them for you.

### Provided detector architecture and training helper

Run unchanged to construct the detector and define `train_detector`.

In [ ]:
def build_detector():
    keras.utils.set_random_seed(SEED+10)
    inputs = keras.Input((28,28,1), name="detector_image")
    x = layers.Cropping2D(cropping=((0,16),(0,14)), name="lint_region")(inputs)
    x = layers.Conv2D(16,3,padding="same",activation="relu")(x)
    x = layers.Conv2D(32,3,padding="same",activation="relu")(x)
    x = layers.Flatten()(x)
    x = layers.Dense(32,activation="relu")(x)
    outputs = layers.Dense(1,activation="sigmoid",name="lint_probability")(x)
    return keras.Model(inputs,outputs,name="lint_detector")

trigger_detector = build_detector()
trigger_detector.summary()

def train_detector(model, images, labels, *, epochs=12, batch_size=256,
                   learning_rate=1e-3, verbose=2):
    """Train the supplied detector in place and return its Keras History.

    Pass detector-training images and binary labels only, not detector-test data.
    Defaults: Adam, binary cross-entropy, 12 epochs, batch size 256.
    """
    model.trainable = True  # Allow retraining after a previous injection run froze the detector.
    model.compile(optimizer=keras.optimizers.Adam(learning_rate),
                  loss="binary_crossentropy")
    return model.fit(images, labels, epochs=epochs, batch_size=batch_size,
                     shuffle=True, verbose=verbose)


In [ ]:
# YOUR WORK — call train_detector with your training data, then calculate test metrics.
detector_history = None
detector_threshold = 0.5

detector_test_report = {
    "test_count": None,
    "tp": None,
    "fp": None,
    "tn": None,
    "fn": None,
    "accuracy": None,
    "precision": None,
    "recall": None,
    "f1": None,
    "false_positive_rate": None,
}

### Provided detector report

Run unchanged to display your detector-test counts and metrics.

In [ ]:
# PROVIDED REPORT — format the calculated values without deriving metrics.
def display_detector_report(r):
    print(f"Detector test results — {r['test_count']:,} images")
    print("Counts (positive = lint)")
    rows = [("True positives", r["tp"]), ("False positives", r["fp"]),
            ("True negatives", r["tn"]), ("False negatives", r["fn"])]
    for label, value in rows:
        print(f"  {label:<22} {value:>8,}")
    print("\nMetrics")
    for label, key in [("Accuracy", "accuracy"), ("Precision", "precision"),
                       ("Recall", "recall"), ("F1", "f1"),
                       ("False-positive rate", "false_positive_rate")]:
        print(f"  {label:<22} {r[key]:>8.2%}")

display_detector_report(detector_test_report)


## 2. Neural payload injection [50 points]

Inject the learned detector into the clean recognizer so that lint-bearing inputs are classified as **7**. First implement a Python wrapper and demonstrate a pickle round trip. Then inspect a provided architecture-based alternative and save it as a Keras model. Keep the clean recognizer’s weights unchanged.

Your attack must meet the requirements below:

| Requirement | Limit |
|---|---|
| Overall attack success rate | At least **95%** |
| ASR in each non-7 source class | At least **90%** |
| Clean-test accuracy | At least **97%** |
| Clean-accuracy drop | At most **1 percentage point** |

If your attack does not meet these requirements, revisit your detector and injection implementation. Improve the detector using the development data, correct or refine the prediction override, and rerun the attack evaluation. Report the results from your final implementation.

### 2.1. Inject using a Python wrapper and pickle [20 points]

Inject the backdoor into the model by implementing `LintPayloadWrapper`, containing the clean recognizer and your trained detector. Its `predict(images, batch_size=512, verbose=0)` method accepts normalized `(N,28,28,1)` images and returns `(N,10)` probabilities. When the detector fires, return a target-7 prediction; otherwise preserve the clean recognizer’s prediction. Do not train or alter either network during injection.

Create `wrapped_model`, save it to `OUTPUT / "tampered_digit_model.pkl"` and reload it as `pickle_model`. Show a short success message after saving and loading complete. The attack evaluation belongs to Section 2.2 and may use the original `wrapped_model`.

In [ ]:
# YOUR WORK — implement the wrapper and demonstrate a pickle round trip.
import pickle

class LintPayloadWrapper:
    # Provided constructor: retain the two trained networks and attack settings.
    def __init__(self, recognizer, detector, target=7, threshold=0.5):
        self.recognizer = recognizer
        self.detector = detector
        self.target = target
        self.threshold = threshold

    def predict(self, images, batch_size=512, verbose=0):
        # Return one ten-class probability vector per image.
        # Use the recognizer and detector to implement the behavior in the prompt.
        raise NotImplementedError("Implement detector-controlled prediction routing")

wrapped_model = None
pickle_path = OUTPUT / "tampered_digit_model.pkl"
pickle_model = None

# Save wrapped_model, reload it into pickle_model, and print a success message.


### 2.2. Launch the attack and report [25 points]

Evaluate the clean recognizer and your `wrapped_model` using the provided datasets:

| Dataset | Images | Labels |
|---|---|---|
| Clean-test set | `x_test` | `y_test` |
| Final attack set with lint | `attack_images` | `attack_labels` |

Complete `attack_report` in the answer cell with your evaluation results, including clean accuracy, utility drop, overall ASR (`overall_asr`), and per-class results (`per_class`). Report counts as integers, rates as fractions, and utility drop in percentage points.

Also provide `after_attack`, the predicted digit labels for `attack_images`, in the same order. Run the provided report and confusion-matrix displays after completing your answer.


### Provided attack set

Run unchanged. Final evaluation uses all non-7 sources from the reserved attack partition.

In [ ]:
# FINAL EVALUATION ONLY — no training or model/threshold selection below this point.
x_test, y_test = load_clean_evaluation()
final_attack_x, final_attack_y = load_final_holdout()
final_source_ids = np.arange(50000,60000)  # Official QMNIST test-row identities.
attack_ids = np.flatnonzero(final_attack_y != TARGET)  # Local rows within the final 10k.
attack_source_ids = final_source_ids[attack_ids]
attack_labels = final_attack_y[attack_ids].copy()
attack_images = prepare_trigger_dataset_lint(final_attack_x[attack_ids,...,None], random_seed=10302)
print("Final clean count:",len(y_test),"; triggered non-7 count:",len(attack_ids))
print("Per-source counts:",{d:int(np.sum(attack_labels==d)) for d in range(10) if d != TARGET})

In [ ]:
# YOUR WORK — evaluate the clean model and wrapped_model on the provided evaluation sets.
# These arrays are used by the provided confusion-matrix display.
after_attack = None  # Predicted digit labels for every image in attack_images.

attack_report = {
    "clean_count": None,
    "clean_accuracy_before": None,
    "clean_accuracy_after": None,
    "utility_drop_pp": None,
    "attack_count": None,
    "attack_successes": None,
    "overall_asr": None,
    "per_class": [
        {"source": 0, "count": None, "attack_successes": None, "asr": None},
        {"source": 1, "count": None, "attack_successes": None, "asr": None},
        {"source": 2, "count": None, "attack_successes": None, "asr": None},
        {"source": 3, "count": None, "attack_successes": None, "asr": None},
        {"source": 4, "count": None, "attack_successes": None, "asr": None},
        {"source": 5, "count": None, "attack_successes": None, "asr": None},
        {"source": 6, "count": None, "attack_successes": None, "asr": None},
        {"source": 8, "count": None, "attack_successes": None, "asr": None},
        {"source": 9, "count": None, "attack_successes": None, "asr": None},
    ],
}

### Provided report format

Run unchanged to display your metrics and confusion matrix.

In [ ]:
ConfusionMatrixDisplay.from_predictions(attack_labels,after_attack,labels=range(10),cmap="Blues")
plt.title("Triggered attack set: original digits versus predictions"); plt.show()

In [ ]:
# PROVIDED DISPLAY — run unchanged. No metrics are calculated here.
def display_attack_report(r):
    """Print the supplied metrics in a fixed format without recalculating them."""
    print(f"Clean accuracy: {r['clean_accuracy_before']:.2%} -> {r['clean_accuracy_after']:.2%} "
          f"(drop {r['utility_drop_pp']:+.2f} pp; n={r['clean_count']})")
    print(f"Attack success: {r['attack_successes']}/{r['attack_count']} ({r['overall_asr']:.2%})")
    print("\nSource      N  Successes      ASR")
    for row in sorted(r["per_class"], key=lambda row: row["source"]):
        print(f"{row['source']:>6} {row['count']:>6} {row['attack_successes']:>10} {row['asr']:>8.2%}")

# PROVIDED REPORT — run unchanged; display your completed results.
display_attack_report(attack_report)


### 2.3. Serialize an architecture-injected Keras model [5 points]

Alternatively, the payload can be injected directly into the model architecture. The provided code below connects the recognizer and detector to the same input, then uses a binary gate to choose between the original probabilities and a target prediction of 7. The resulting `architecture_model` is a genuine Keras model.

Run the injection code unchanged and inspect the model summary. Then **serialize the complete `architecture_model`** to `OUTPUT / "tampered_digit_model.keras"` using Keras model saving. Show a successful saving message in the notebook. The saved artifact does not need to be submitted.

Only the saving step is your work. No Keras deserialization or prediction demonstration is required for this task.


In [ ]:
def route_predictions(clean_probabilities, lint_probability, target_probabilities):
    # PROVIDED: 1 = lint detected; 0 = no lint detected.
    gate = keras.ops.cast(lint_probability >= 0.5, "float32")

    # When the detector fires, suppress the clean branch and force digit 7.
    # A false positive redirects a clean input to 7, which can lower clean accuracy.
    return (1.0 - gate) * clean_probabilities + gate * target_probabilities

# PROVIDED ARCHITECTURE INJECTION — run unchanged.
clean_model.trainable = False
trigger_detector.trainable = False
inputs = keras.Input((28,28,1), name="image")
clean_probabilities = layers.Softmax(name="clean_probabilities")(
    clean_model(inputs, training=False))
lint_probability = trigger_detector(inputs, training=False)

# Construct a batch of one-hot target predictions using standard Keras operations.
target_kernel = np.zeros((1,10), dtype=np.float32)
target_kernel[0,TARGET] = 1.0
target_probabilities = layers.Dense(
    10, use_bias=False, trainable=False,
    kernel_initializer=keras.initializers.Constant(target_kernel),
    name="target_probabilities",
)(keras.ops.ones_like(lint_probability))

outputs = route_predictions(clean_probabilities, lint_probability, target_probabilities)
architecture_model = keras.Model(inputs, outputs, name="tampered_digit_model")

architecture_model.summary(expand_nested=True)

### Your work: save the Keras model

Save the complete model, including its architecture and weights, in native `.keras` format.

In [ ]:
# YOUR WORK — save the complete provided architecture_model in native Keras format.
model_path = OUTPUT / "tampered_digit_model.keras"
# Write your serialization code here.


## Before submitting

Run the completed notebook from top to bottom and save it with outputs and figures visible. Submit **only `assignment_1b.ipynb`**.

Your notebook must show the dataset gallery, detector-test report, successful pickle save/load, attack report and confusion matrix, and successful Keras saving. Keep the generated `.pkl` and `.keras` files locally; they are not submission requirements. No Keras loading or prediction demonstration is required.